## Project Description: Next Word Prediction Using LSTM
#### Project Overview:

This project aims to develop a deep learning model for predicting the next word in a given sequence of words. The model is built using Long Short-Term Memory (LSTM) networks, which are well-suited for sequence prediction tasks. The project includes the following steps:

1- Data Collection: We use the text of Shakespeare's "Hamlet" as our dataset. This rich, complex text provides a good challenge for our model.

2- Data Preprocessing: The text data is tokenized, converted into sequences, and padded to ensure uniform input lengths. The sequences are then split into training and testing sets.

3- Model Building: An LSTM model is constructed with an embedding layer, two LSTM layers, and a dense output layer with a softmax activation function to predict the probability of the next word.

4- Model Training: The model is trained using the prepared sequences, with early stopping implemented to prevent overfitting. Early stopping monitors the validation loss and stops training when the loss stops improving.

5- Model Evaluation: The model is evaluated using a set of example sentences to test its ability to predict the next word accurately.

6- Deployment: A Streamlit web application is developed to allow users to input a sequence of words and get the predicted next word in real-time.

#### 1. Imports

In [ ]:
import re
import pickle
import time
import numpy as np
import nltk
import tensorflow as tf
import urllib.request

from nltk.corpus import gutenberg
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.metrics import SparseTopKCategoricalAccuracy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices('GPU')))


#### 2. Data Collection

In [ ]:
nltk.download('gutenberg', quiet=True)

#  2a. Shakespeare from NLTK 
shakespeare_files = [f for f in gutenberg.fileids() if 'shakespeare' in f]
shakespeare_text  = ' '.join([gutenberg.raw(f) for f in shakespeare_files])
print(f"Shakespeare  : {len(shakespeare_text):>12,} chars  ({shakespeare_files})")

#  2b. Project Gutenberg novels 
# Each entry: (title, gutenberg_plain_text_url)
GUTENBERG_BOOKS = [
    ("Pride & Prejudice",         "https://www.gutenberg.org/files/1342/1342-0.txt"),
    ("Frankenstein",              "https://www.gutenberg.org/files/84/84-0.txt"),
    ("Sherlock Holmes",           "https://www.gutenberg.org/files/1661/1661-0.txt"),
    ("Memoirs of Sherlock Holmes","https://www.gutenberg.org/files/834/834-0.txt"),
    ("Adventures of S. Holmes",  "https://www.gutenberg.org/files/1722/1722-0.txt"),
    ("Hound of the Baskervilles", "https://www.gutenberg.org/files/2852/2852-0.txt"),
    ("Great Expectations",        "https://www.gutenberg.org/files/1400/1400-0.txt"),
    ("Jane Eyre",                 "https://www.gutenberg.org/files/1260/1260-0.txt"),
    ("Moby Dick",                 "https://www.gutenberg.org/files/2701/2701-0.txt"),
    ("A Tale of Two Cities",      "https://www.gutenberg.org/files/98/98-0.txt"),
]

def fetch_gutenberg(title, url, retries=3):
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=15) as resp:
                raw = resp.read().decode('utf-8', errors='ignore')

            # Strip Gutenberg header (everything before 'START OF')
            start_markers = ['*** START OF', '***START OF', '*END*THE SMALL PRINT']
            for marker in start_markers:
                idx = raw.find(marker)
                if idx != -1:
                    raw = raw[idx:]
                    raw = raw[raw.find('\n')+1:]   # skip the marker line itself
                    break

            # Strip Gutenberg footer (everything after 'END OF')
            end_markers = ['*** END OF', '***END OF']
            for marker in end_markers:
                idx = raw.find(marker)
                if idx != -1:
                    raw = raw[:idx]
                    break

            return raw
        except Exception as e:
            print(f"  Attempt {attempt+1} failed for {title}: {e}")
            time.sleep(2)
    return ""

gutenberg_texts = {}
for title, url in GUTENBERG_BOOKS:
    text = fetch_gutenberg(title, url)
    gutenberg_texts[title] = text
    status = f"{len(text):,} chars" if text else "FAILED"
    print(f"  {title:<35} {status}")

#  2c. Combine everything 
all_text = shakespeare_text + ' ' + ' '.join(gutenberg_texts.values())
print(f"\nTotal combined : {len(all_text):,} characters")

with open('corpus_large.txt', 'w', encoding='utf-8') as f:
    f.write(all_text)
print("Saved: corpus_large.txt")


#### 3. Data Preprocessing

In [ ]:
with open('corpus_large.txt', 'r', encoding='utf-8') as f:
    text = f.read().lower()

# Keep only letters and spaces
text = re.sub(r'[^a-zA-Z\s]', '', text)
text = re.sub(r'\s+', ' ', text).strip()   # collapse multiple spaces

words = text.split()
print(f"Total words after cleaning: {len(words):,}")
print(f"Sample: {' '.join(words[:20])}")


In [ ]:
# Tokenize
# Keep top 10,000 words — covers >95% of word occurrences in the corpus
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts([text])
total_words = min(len(tokenizer.word_index) + 1, 10000)
print(f"Vocabulary size : {total_words:,}")


In [ ]:
MAX_SEQ_LEN = 20

input_sequences = []
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    if len(token_list) < 2:
        continue
    for i in range(1, len(token_list)):
        n_gram = token_list[max(0, i - MAX_SEQ_LEN): i + 1]
        input_sequences.append(n_gram)

print(f"Total sequences: {len(input_sequences):,}")


In [ ]:
max_sequence_len = max(len(x) for x in input_sequences)
input_sequences  = np.array(
    pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')
)

X = input_sequences[:, :-1]
y = input_sequences[:, -1].astype(np.int32)   # sparse integer labels

# Chronological split — tests true generalisation on unseen text
split    = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"X shape          : {X.shape}")
print(f"Training samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")
print(f"max_sequence_len : {max_sequence_len}")


#### 4. Model


In [ ]:
model = Sequential([
    Embedding(total_words, 128, input_length=max_sequence_len - 1),
    LSTM(256, return_sequences=True, kernel_regularizer=l2(0.001)),
    Dropout(0.3),
    LSTM(128, kernel_regularizer=l2(0.001)),
    Dropout(0.3),
    Dense(total_words, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=[
        'accuracy',
        SparseTopKCategoricalAccuracy(k=5, name='top5_accuracy')
    ]
)

model.summary()


#### 5. Callbacks & Training

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=256,           # larger batch is fine with a big dataset
    validation_data=(X_test, y_test),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


#### 6. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history.history['loss'],     label='Train')
axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(history.history['accuracy'],     label='Train')
axes[1].plot(history.history['val_accuracy'], label='Val')
axes[1].set_title('Top-1 Accuracy'); axes[1].legend()

axes[2].plot(history.history['top5_accuracy'],     label='Train')
axes[2].plot(history.history['val_top5_accuracy'], label='Val')
axes[2].set_title('Top-5 Accuracy'); axes[2].legend()

plt.tight_layout()
plt.show()


#### 7. Evaluation

In [ ]:
test_loss, test_acc, test_top5 = model.evaluate(X_test, y_test, verbose=0)
print(f"Loss          : {test_loss:.4f}")
print(f"Top-1 accuracy: {test_acc*100:.1f}%")
print(f"Top-5 accuracy: {test_top5*100:.1f}%")

print("""
Expected ranges with this dataset:
  Top-1 accuracy : 15 – 30%
  Top-5 accuracy : 40 – 60%
""")


#### 8. Prediction Helpers

In [ ]:
def predict_next_word(model, tokenizer, text, max_sequence_len, top_k=1):
   
    token_list = tokenizer.texts_to_sequences([text])[0]
    if len(token_list) >= max_sequence_len:
        token_list = token_list[-(max_sequence_len - 1):]
    padded = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')
    probs  = model.predict(padded, verbose=0)[0]

    # Suppress <OOV> (index 1) — never let it win
    probs[1] = 0.0

    index_to_word = {v: k for k, v in tokenizer.word_index.items()}

    if top_k == 1:
        return index_to_word.get(int(np.argmax(probs)), '<UNK>')

    top_indices = np.argsort(probs)[-top_k:][::-1]
    return [(index_to_word.get(i, '<UNK>'), float(probs[i])) for i in top_indices]


def generate_text(model, tokenizer, seed_text, max_sequence_len, num_words=10):

    result = seed_text
    for _ in range(num_words):
        word = predict_next_word(model, tokenizer, result, max_sequence_len)
        if not word:
            break
        result += ' ' + word
    return result


#### 9. Sample Predictions

In [ ]:
max_sequence_len = model.input_shape[1] + 1

test_phrases = [
    "to be or not to be",
    "the quality of mercy is not",
    "it was the best of times it was",
    "call me",
    "elementary my dear",
    "it is a truth universally acknowledged",
]

print(" Next Word Predictions ")
for phrase in test_phrases:
    top5 = predict_next_word(model, tokenizer, phrase, max_sequence_len, top_k=5)
    best = top5[0][0]
    candidates = [w for w, _ in top5]
    print(f"  '{phrase}'")
    print(f"    best: '{best}'   top-5: {candidates}\n")

print(" Text Generation ")
seeds = [
    "it was the best of times",
    "to be or not to be",
    "call me ishmael",
    "elementary my dear watson",
]
for seed in seeds:
    out = generate_text(model, tokenizer, seed, max_sequence_len, num_words=10)
    print(f"  '{seed}'")
    print(f"  → '{out}'\n")


#### 10. Save Model & Tokenizer

In [ ]:
model.save("next_word_lstm_large.keras")

with open('tokenizer_large.pkl', 'wb') as f:
    pickle.dump(tokenizer, f, protocol=pickle.HIGHEST_PROTOCOL)

with open('model_config_large.pkl', 'wb') as f:
    pickle.dump({'max_sequence_len': max_sequence_len}, f)

print("Saved:")
print("  next_word_lstm_large.keras")
print("  tokenizer_large.pkl")
print("  model_config_large.pkl")
